In [6]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("/workspaces/RAG-/He_Deep_Residual_Learning_CVPR_2016_paper.pdf")
data=loader.load()

In [9]:
data

print(data[0].page_content)

Deep Residual Learning for Image Recognition
Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun
Microsoft Research
{kahe, v-xiangz, v-shren, jiansun }@microsoft.com
Abstract
Deeper neural networks are more difﬁcult to train. We
present a residual learning framework to ease the training
of networks that are substantially deeper than those used
previously. We explicitly reformulate the layers as learn-
ing residual functions with reference to the layer inputs, in-
stead of learning unreferenced functions. We provide com-
prehensive empirical evidence showing that these residual
networks are easier to optimize, and can gain accuracy from
considerably increased depth. On the ImageNet dataset we
evaluate residual nets with a depth of up to 152 layers—8 ×
deeper than VGG nets [40] but still having lower complex-
ity. An ensemble of these residual nets achieves 3.57% error
on the ImageNet test set. This result won the 1st place on the
ILSVRC 2015 classiﬁcation task. We also present analysis
on CI

In [8]:
len(data)

9

In [11]:
%pip install langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs = text_splitter.split_documents(data)

print("total number of documents", len(docs))

Note: you may need to restart the kernel to use updated packages.
total number of documents 58


In [14]:
from langchain_chroma import Chroma
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

from dotenv import load_dotenv
import os
import getpass
load_dotenv()

api_key = os.getenv("NVIDIA_API_KEY")
if not api_key:
    api_key = getpass.getpass("Enter NVIDIA_API_KEY (input hidden): ").strip()

if api_key:
    os.environ["NVIDIA_API_KEY"] = api_key
    embeddings = NVIDIAEmbeddings(
        model="nvidia/nv-embed-v1",
        api_key=api_key
    )
    vector = embeddings.embed_query("hello, world")
    vector[:5]
else:
    print("No API key provided. Set NVIDIA_API_KEY to run embeddings.")

In [15]:
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)

In [17]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})
retriever_docs=retriever.invoke("what is the embedding method in building a LLM")

In [19]:
retrieved_docs = retriever_docs
len(retrieved_docs)

10

In [20]:
print(retrieved_docs[5].page_content)

P . Doll´ar, and C. L. Zitnick. Microsoft COCO: Common objects in
context. In ECCV. 2014.
[27] J. Long, E. Shelhamer, and T. Darrell. Fully convolutional networks
for semantic segmentation. In CVPR, 2015.
[28] G. Mont ´ufar, R. Pascanu, K. Cho, and Y . Bengio. On the number of
linear regions of deep neural networks. In NIPS, 2014.
[29] V . Nair and G. E. Hinton. Rectiﬁed linear units improve restricted
boltzmann machines. In ICML, 2010.
[30] F. Perronnin and C. Dance. Fisher kernels on visual vocabularies for
image categorization. In CVPR, 2007.
[31] T. Raiko, H. V alpola, and Y . LeCun. Deep learning made easier by
linear transformations in perceptrons. In AISTATS, 2012.
[32] S. Ren, K. He, R. Girshick, and J. Sun. Faster R-CNN: Towards
real-time object detection with region proposal networks. In NIPS,
2015.
[33] B. D. Ripley. Pattern recognition and neural networks . Cambridge
university press, 1996.
[34] A. Romero, N. Ballas, S. E. Kahou, A. Chassang, C. Gatta, and


In [21]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0.3, max_tokens=500)

/tmp/ipykernel_14300/181523164.py:2: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0.3, max_tokens=500)


In [25]:
%pip install -q -U langchain langchain-classic

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "you are an assistant for a question-answering task. "
    "use the following pieces of retrieved context to answer "
    "the question. If you dont know the answer, say that you dont know. "
    "use three sentences maximum and keep the "
    "answer concise"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

query = input("Enter your question: ")
if query:
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    response = rag_chain.invoke({"input": query})
    print(response["answer"])

Note: you may need to restart the kernel to use updated packages.
